# 신뢰구간과 가설검정

> 파이썬 13강 · 통계적 추론

이 노트북은 웹 강의의 **실습 부분만** 옮겨온 것입니다.
자세한 설명과 그림은 원문을 함께 보세요 → [신뢰구간과 가설검정](https://mioon1402.github.io/timeseriesdata/python/p13-inference.html)

---

**먼저 아래 준비 셀을 한 번 실행하세요.** 예시 데이터를 내려받습니다.

In [ ]:
# 예시 데이터 내려받기
!wget -q -nc https://raw.githubusercontent.com/mioon1402/timeseriesdata/main/data/cafe_sales.csv

# 그래프 한글 깨짐 방지
!pip install -q koreanize-matplotlib
import koreanize_matplotlib  # noqa: F401

print('준비 완료')

## 1. 평균의 신뢰구간

**13-1. 신뢰구간 구하기**

In [ ]:
import pandas as pd
from scipy import stats

df = pd.read_csv("cafe_sales.csv", parse_dates=["date"])
v = df["visitors"]

lo, hi = stats.t.interval(0.95, len(v) - 1,
                          loc=v.mean(), scale=stats.sem(v))

print(f"표본평균  {v.mean():.1f}명   (n = {len(v)})")
print(f"표준오차  {stats.sem(v):.2f}")
print(f"95% 신뢰구간  [{lo:.1f}, {hi:.1f}]")
print(f"오차범위  ±{(hi - lo) / 2:.1f}명")

## 2. 두 그룹 비교 — t검정

**13-2. 주말 vs 평일**

In [ ]:
df["주말"] = df["date"].dt.dayofweek >= 5
주말 = df[df["주말"]]["visitors"]
평일 = df[~df["주말"]]["visitors"]

print(f"주말  n={len(주말):3d}  평균 {주말.mean():6.1f}  SD {주말.std():5.1f}")
print(f"평일  n={len(평일):3d}  평균 {평일.mean():6.1f}  SD {평일.std():5.1f}")
print(f"차이  {주말.mean() - 평일.mean():+.1f}명")
print()

t, p = stats.ttest_ind(주말, 평일, equal_var=False)   # Welch t검정
print(f"t = {t:.3f}")
print(f"p = {p:.4g}")

## 3. p값보다 중요한 것 — 차이의 신뢰구간

**13-3. 차이의 95% 신뢰구간**

In [ ]:
import numpy as np

n1, n2 = len(주말), len(평일)
var1, var2 = 주말.var(), 평일.var()

차이 = 주말.mean() - 평일.mean()
se = np.sqrt(var1/n1 + var2/n2)

# Welch-Satterthwaite 자유도
자유도 = (var1/n1 + var2/n2)**2 / ((var1/n1)**2/(n1-1) + (var2/n2)**2/(n2-1))
임계값 = stats.t.ppf(0.975, 자유도)

print(f"차이        {차이:.1f}명")
print(f"표준오차    {se:.2f}")
print(f"95% 신뢰구간 [{차이 - 임계값*se:.1f}, {차이 + 임계값*se:.1f}]")

## 4. 효과크기 — 차이가 얼마나 큰가

**13-4. Cohen's d**

In [ ]:
합동SD = np.sqrt(((n1-1)*var1 + (n2-1)*var2) / (n1 + n2 - 2))
d = (주말.mean() - 평일.mean()) / 합동SD

print(f"합동 표준편차 {합동SD:.1f}")
print(f"Cohen's d   {d:.3f}")
print()
크기 = "작음" if abs(d) < 0.5 else "중간" if abs(d) < 0.8 else "큼"
print(f"→ 효과크기 '{크기}'")

## 5. 순열검정 — 가정 없이 하기

**13-5. 순열검정 직접 만들기**

In [ ]:
rng = np.random.default_rng(42)

관측차이 = abs(주말.mean() - 평일.mean())
전체 = np.concatenate([주말.values, 평일.values])
n = len(주말)

N = 5000
극단 = 0
for _ in range(N):
    rng.shuffle(전체)                       # 라벨을 떼고 섞는다
    섞인차이 = abs(전체[:n].mean() - 전체[n:].mean())
    if 섞인차이 >= 관측차이:
        극단 += 1

p_순열 = (극단 + 1) / (N + 1)
print(f"관측된 차이        {관측차이:.1f}명")
print(f"섞어서 더 극단적    {극단}회 / {N}회")
print(f"순열검정 p          {p_순열:.5f}")
print(f"t검정 p             {p:.4g}")

## 6. 여러 번 검정할 때

**13-6. 다중검정 보정**

In [ ]:
from itertools import combinations

순서 = ["월", "화", "수", "목", "금", "토", "일"]
그룹 = {d: df[df["weekday"] == d]["visitors"] for d in 순서}

결과 = []
for a, b in combinations(순서, 2):
    _, pv = stats.ttest_ind(그룹[a], 그룹[b], equal_var=False)
    결과.append((f"{a}-{b}", pv))

유의 = sum(1 for _, pv in 결과 if pv < 0.05)
본페로니 = 0.05 / len(결과)
유의보정 = sum(1 for _, pv in 결과 if pv < 본페로니)

print(f"검정 횟수: {len(결과)}회")
print(f"보정 없이 p<0.05:        {유의}쌍")
print(f"본페로니 기준 p<{본페로니:.5f}: {유의보정}쌍")
print()
print("가장 작은 p값 3개:")
for 이름, pv in sorted(결과, key=lambda x: x[1])[:3]:
    print(f"  {이름}  p={pv:.3g}")

## 7. 보고서에 쓰는 법

**연습 · 직접 써보세요**

In [ ]:
# 문제 1. 비 온 날과 안 온 날의 방문객을 t검정으로 비교하고,
#        차이의 신뢰구간과 Cohen's d 까지 구해보세요.


# 문제 2. 공휴일과 그 외의 방문객을 비교해보세요.
#        공휴일은 35일뿐인데, 신뢰구간의 폭이 어떻게 다른가요?


# 문제 3. 계절 4개 그룹의 매출을 ANOVA(f_oneway)로 비교해보세요.

**모범 답안**

In [ ]:
def 비교(이름, a, b):
    t, pv = stats.ttest_ind(a, b, equal_var=False)
    n1, n2 = len(a), len(b)
    v1, v2 = a.var(), b.var()
    차 = a.mean() - b.mean()
    se = np.sqrt(v1/n1 + v2/n2)
    fr = (v1/n1 + v2/n2)**2 / ((v1/n1)**2/(n1-1) + (v2/n2)**2/(n2-1))
    tc = stats.t.ppf(0.975, fr)
    sp = np.sqrt(((n1-1)*v1 + (n2-1)*v2) / (n1+n2-2))
    print(f"[{이름}]  n={n1} vs {n2}")
    print(f"  차이 {차:+.1f}  95% CI [{차-tc*se:.1f}, {차+tc*se:.1f}]")
    print(f"  d={차/sp:.3f}  p={pv:.4g}\n")

# 문제 1
비교("비 vs 맑음", df[df["rain_mm"] > 0]["visitors"], df[df["rain_mm"] == 0]["visitors"])

# 문제 2
비교("공휴일 vs 그 외", df[df["is_holiday"]]["visitors"], df[~df["is_holiday"]]["visitors"])

# 문제 3
def 계절(m):
    return "봄" if m in (3,4,5) else "여름" if m in (6,7,8) else "가을" if m in (9,10,11) else "겨울"
df["계절"] = df["date"].dt.month.map(계절)
그룹들 = [df[df["계절"] == s]["sales"].dropna() for s in ["봄", "여름", "가을", "겨울"]]
F, pv = stats.f_oneway(*그룹들)
print(f"[ANOVA] F={F:.2f}, p={pv:.3g}")
print("→ 적어도 한 계절은 다르다. 어느 쌍인지는 사후검정으로 확인해야 함")

---

전체 강의 목록 → [눈으로 보는 통계](https://mioon1402.github.io/timeseriesdata/)